# 06 Managerial Control

This notebook uses GET only. It reads your watchlists, saved screens, and alerts, plus the Flow Capital fund portfolio summary, once each. It never changes them.

These are member research routes (beta); they are not yet listed in `/api/v1/catalog`.

Research data only, not investment advice. Each panel refreshes on a schedule; each panel shows its source date.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_panel, get_api_key, make_client
client, catalogue = make_client()
api_key = get_api_key()

In [ ]:
watchlist_columns = {
    'watchlist_id': 'The identifier for your saved list.',
    'name': 'The name you gave the saved list.',
    'updated_at': 'The latest item time recorded for the list.',
    'market': 'The market attached to the saved ticker.',
    'ticker': 'The ticker saved in the list.',
    'note': 'Your note attached to the ticker.',
    'added_at': 'The time when the ticker joined the list.',
    'sort_order': 'Your stored display order for the ticker.',
}
def watchlist_rows(payload):
    return [{'watchlist_id': item.get('watchlist_id'), 'name': item.get('name'), 'updated_at': item.get('updated_at'), **(entry or {})} for item in payload.get('watchlists') or [] for entry in item.get('items') or [None]]
watchlists_frame = fetch_panel('Your watchlists', lambda: client.member_watchlists(api_key), watchlist_rows, watchlist_columns, source_date=lambda p: max((item.get('updated_at') for item in p.get('watchlists') or [] if item.get('updated_at')), default=None), status=lambda p: 'degraded' if p.get('degraded') else 'not supplied', message=lambda p: 'The service reported a degraded watchlist read.' if p.get('degraded') else (None if p.get('watchlists') else 'No saved watchlists returned.'))

In [ ]:
screen_columns = {
    'screen_id': 'The identifier for your saved screen.',
    'name': 'The name you gave the screen.',
    'market': 'The market selected for the screen.',
    'tab': 'The saved screen tab.',
    'filters': 'The saved filter object, encoded as JSON.',
    'sort_field': 'The field used to sort screen rows.',
    'sort_dir': 'The saved sort direction.',
    'created_at': 'The creation time for the saved definition.',
}
screens_frame = fetch_panel('Your saved screens', lambda: client.member_screens(api_key), lambda p: p.get('screens') or [], screen_columns, source_date=lambda p: max((row.get('created_at') for row in p.get('screens') or [] if row.get('created_at')), default=None), status=lambda p: 'not supplied', message=lambda p: None if p.get('screens') else 'No saved screens returned.')

In [ ]:
alert_columns = {
    'alert_id': 'The identifier for your alert.',
    'alert_type': 'The type of condition stored for the alert.',
    'market': 'The market attached to the alert.',
    'ticker': 'The ticker attached to the alert.',
    'condition': 'The comparison used by the alert evaluator.',
    'threshold': 'The numeric threshold stored for the alert.',
    'created_at': 'The creation time for the alert.',
    'active': 'Whether the alert is active.',
    'note': 'Your note attached to the alert.',
}
alerts_frame = fetch_panel('Your alerts', lambda: client.member_alerts(api_key), lambda p: p.get('alerts') or [], alert_columns, source_date=lambda p: max((row.get('created_at') for row in p.get('alerts') or [] if row.get('created_at')), default=None), status=lambda p: 'not supplied', message=lambda p: None if p.get('alerts') else 'No alerts returned.')

In [ ]:
summary_columns = {
    'as_of_date': 'The holdings date of the Flow Capital house fund; empty for a cash-only baseline.',
    'nav_usd': 'Fund net asset value in US dollars.',
    'cash_pct': 'Share of net asset value held as cash.',
    'invested_pct': 'Share of net asset value held in positions.',
    'n_positions': 'Number of non-cash positions.',
    'state': 'Set when the fund holds only its starting cash.',
}
def summary_rows(payload):
    return [payload] if payload.get('nav_usd') is not None else []
summary_frame = fetch_panel('Flow Capital fund portfolio summary (house fund, not your own holdings)', lambda: client.member_fund_portfolio_summary(api_key), summary_rows, summary_columns, source_date=lambda p: p.get('as_of_date'), status=lambda p: p.get('state') or 'not supplied', message=lambda p: None if p.get('nav_usd') is not None else 'The service returned no fund summary.')